In [ ]:
import torch
import numpy as np
from scipy.stats import norm
from tqdm.autonotebook import tqdm
from IPython.display import clear_output
from pathlib import Path
import csv

# gpu + experimental settings
device = "cuda" if torch.cuda.is_available() else "cpu"
N_SEEDS, TAU, ALPHA = 100, 0.5, 0.05
BUDGET_PROPS = np.round(np.linspace(0.0, 0.25, 11)[1:], decimals=3)
dataset = "mmlu-pro"  # or "bbh+gpqa+ifeval+math+musr"

METHOD = "ppat"
LAMBDA_INIT, LAMBDA_UPDATE_EVERY = 0.5, 100  # PPAT Section 5.1, p. 9
DATA_DIR = Path(f"../datasets/llm_evals/{dataset}")
LOG_DIR = Path("../logs")
fname = f"{dataset}_{METHOD}.csv"
log_path = LOG_DIR / fname
lambda_path = LOG_DIR / f"{dataset}_{METHOD}_lambda.csv"

# preserve the original per-model, per-seed schema and column order.
columns = ["dataset", "budget", "tau", "method", "model_idx",
           "seed", "thetahat", "coverage", "width", "n_labels"]
lambda_columns = ["dataset", "budget", "tau", "method", "model_idx", "seed", "lambda_hat", "n_labels"]
OVERWRITE = False  # Set True deliberately, or choose a fresh LOG_DIR. No silent append/restart.

In [ ]:
#### LOAD IN OUR DATA

# same factor-model inputs and layout as PAI
U = torch.load(DATA_DIR / "U.pt", map_location=device, weights_only=True).to(device)  # (N_OLD,D)
V = torch.load(DATA_DIR / "V.pt", map_location=device, weights_only=True).to(device)  # (N_Q,D)
ys = torch.load(DATA_DIR / "Y_test.pt", map_location=device, weights_only=True).to(device)  # (N_NEW,N_Q)
if U.ndim != 2 or V.ndim != 2 or ys.ndim != 2:
    raise ValueError("Expected U, V, and Y_test all to be two-dimensional.")
N_NEW, N_Q = ys.shape
D = U.shape[1]
if U.shape[0] < 2 or N_NEW < 1 or D < 1 or V.shape != (N_Q, D):
    raise ValueError("Require >=2 historical models, >=1 target model, and V.shape == (N_Q,D).")
if U.dtype != V.dtype or U.dtype not in (torch.float32, torch.float64):
    raise ValueError("U and V must have the same float32/float64 dtype; model arithmetic is not recast.")
if not torch.isfinite(U).all().item() or not torch.isfinite(V).all().item():
    raise ValueError("Factor tensors must be finite.")
if not ((ys == 0) | (ys == 1)).all().item():
    raise ValueError("Y_test must contain binary correctness labels, without missing entries.")
if N_SEEDS < 1 or not (0 < TAU <= 1) or not (0 < ALPHA < 1):
    raise ValueError("Require N_SEEDS >= 1, 0 < TAU <= 1, and 0 < ALPHA < 1.")
if LAMBDA_UPDATE_EVERY < 1:
    raise ValueError("LAMBDA_UPDATE_EVERY must be positive.")
if not all(2 <= int(b * N_Q) < N_Q for b in BUDGET_PROPS):
    raise ValueError("Each budget must yield 2 <= n < population size for these CIs.")

# Preserve the original initialization, including torch.std's sample-std convention.
initial_factor = U.mean(axis=0, keepdim=True)
initial_precision = 1.0 / (U.std(axis=0, keepdim=True) ** 2)
if not torch.isfinite(initial_precision).all().item() or not (initial_precision > 0).all().item():
    raise ValueError("U must have a finite, positive sample variance in every factor dimension; no new prior floor is added.")

# PPAT Section 3: use the initial factor prediction as the FIXED soft proxy.
# All target models share this initial proxy, but their acquired labels, lambda estimates,
# surrogate factor vectors, and selected questions remain separate.
with torch.no_grad():
    proxy_fxs = torch.sigmoid(initial_factor @ V.T).squeeze(0).to(torch.float64).clone()
if not torch.isfinite(proxy_fxs).all().item():
    raise FloatingPointError("Initial proxy prediction is nonfinite.")
proxy_constant = (proxy_fxs == proxy_fxs[0]).all().item()
cs = proxy_fxs - proxy_fxs.mean()  # (N_Q,), centered over the full question bank
if proxy_constant:
    cs.zero_()
proxy_second_moment = cs.square().mean()  # Eq. (14), known from covariates/initial factors only
c_for_sampling = cs.to(U.dtype).unsqueeze(0)  # (1,N_Q)
if proxy_constant:
    print("Constant proxy: Eq. (14) is undefined; the zero control variate gives LURE (lambda=0).")
elif proxy_second_moment.item() == 0.0:
    raise FloatingPointError("Nonconstant proxy variance underflowed; no denominator floor is applied.")

# For coverage/error reporting ONLY; never used by acquisition or coefficient fitting.
true_thetas = (ys if ys.is_floating_point() else ys.to(U.dtype)).mean(dim=-1, keepdim=True)
print(f"{METHOD}: {dataset}, targets={N_NEW:,}, questions={N_Q:,}, D={D}, seeds={N_SEEDS}, device={device}")
print("Budgets (proportion, labels per target):", [(float(b), int(b * N_Q)) for b in BUDGET_PROPS])
print(f"Nominal coverage={1-ALPHA:.0%}; uniform mixing={TAU}; acquisition factors update every label.")


In [ ]:
#### PREDICTION-POWERED ACTIVE TESTING: fitted lambda_M only

# Fail rather than silently append duplicate experiments on a rerun.
LOG_DIR.mkdir(parents=True, exist_ok=True)
for output_path in (log_path, lambda_path):
    if output_path.exists() and not OVERWRITE:
        raise FileExistsError(f"{output_path} exists. Use a fresh LOG_DIR or deliberately set OVERWRITE=True.")
with open(log_path, "w", newline="") as file:
    csv.writer(file).writerow(columns)
with open(lambda_path, "w", newline="") as file:
    csv.writer(file).writerow(lambda_columns)

# Preserve budget -> seed -> acquisition loops, batching over target models.
# equal RNG seeds are reused across budgets.
with torch.no_grad():
    for budget_idx, budget in enumerate(BUDGET_PROPS):
        n = int(budget * N_Q)
        for seed in range(N_SEEDS):
            torch.random.manual_seed(858 + seed)

            #### initialize data structures
            already_sampled = torch.zeros(size=(N_NEW, N_Q), dtype=bool, device=device)
            # Only queried indices and their ORIGINAL conditional sampling probabilities are retained.
            I_history = torch.empty((N_NEW, n), dtype=torch.long, device=device)
            q_history = torch.empty((N_NEW, n), dtype=torch.float64, device=device)
            lambda_acq = torch.full((N_NEW, 1), 0.0 if proxy_constant else LAMBDA_INIT,
                                    dtype=torch.float64, device=device)
            # EXACT original factor initialization and diagonal precision; V remains frozen.
            Uhats = torch.tile(U.mean(axis=0, keepdim=True), (N_NEW, 1))
            diag_precisions = torch.tile(1.0 / (U.std(axis=0, keepdim=True) ** 2), (N_NEW, 1))

            #### adaptive sampling rounds
            for t in tqdm(range(n), desc=f"Budget {budget_idx + 1} of {len(BUDGET_PROPS)}, Seed {seed} of {N_SEEDS}"):
                #### PART 1: PPAT's binary RMS acquisition, Eqs. (12)-(13), p. 6
                logits = Uhats @ V.T  # (N_NEW,N_Q)
                fxs = torch.sigmoid(logits)
                # sqrt(p*(1-p) + (p-lambda*c)^2), with no labels used in the scores.
                shift = lambda_acq.to(U.dtype) * c_for_sampling
                scores = torch.hypot(torch.sqrt(fxs * (1.0 - fxs)), fxs - shift)
                log_u_ts = scores.log().masked_fill(already_sampled, -torch.inf)
                # All-zero scores: the score-normalized proposal is undefined. Use uniform there.
                zero_score_rows = torch.isneginf(log_u_ts).all(dim=1, keepdim=True)
                log_u_ts = torch.where(zero_score_rows & ~already_sampled, 0.0, log_u_ts)
                h_js = torch.softmax(log_u_ts, dim=1)
                q_js = (TAU * (~already_sampled).to(h_js.dtype) / (N_Q - t)) + ((1.0 - TAU) * h_js)
                q_js = q_js / q_js.sum(dim=1, keepdim=True)  # numerical normalization
                I_t = torch.multinomial(input=q_js, num_samples=1)  # (N_NEW,1)

                #### PART 2: query labels, save probabilities, and remove selected entries
                y_Is = torch.gather(ys, dim=1, index=I_t)  # (N_NEW,1), one query per model
                f_Is = torch.gather(fxs, dim=1, index=I_t)
                q_Is = torch.gather(q_js, dim=1, index=I_t)
                I_history[:, t:t+1] = I_t
                q_history[:, t:t+1] = q_Is.to(torch.float64)
                already_sampled.scatter_(dim=1, index=I_t,
                    src=torch.full_like(I_t, fill_value=True, dtype=bool))

                #### PART 3: original every-label factor update (same update order and prior)
                w_Is = f_Is * (1.0 - f_Is)
                X_Is = V[I_t.squeeze(1)]  # squeeze only the singleton index axis; safe for N_NEW=1
                diag_precisions += w_Is * (X_Is ** 2)
                Uhats += (y_Is.to(f_Is.dtype) - f_Is) * X_Is / diag_precisions

                #### PART 3b: update lambda after each 100 labels for FUTURE draws only
                m = t + 1
                if m % LAMBDA_UPDATE_EVERY == 0 and m < n and not proxy_constant:
                    # Eq. (14), p. 7. Recompute V_{m,s} for CURRENT prefix budget m.
                    # Never replace stored q_s(I_s) by probabilities from the newly updated model.
                    past_ms = torch.arange(1, m + 1, device=device, dtype=torch.float64).unsqueeze(0)
                    V_current = 1.0 + ((N_Q - m) / (N_Q - past_ms)) * (
                        1.0 / ((N_Q - past_ms + 1.0) * q_history[:, :m]) - 1.0)
                    past_indices = I_history[:, :m]
                    past_ys = torch.gather(ys, dim=1, index=past_indices)
                    numerator = (V_current * past_ys.to(torch.float64) * cs[past_indices]).mean(dim=1, keepdim=True)
                    lambda_acq = numerator / proxy_second_moment
                    if not torch.isfinite(lambda_acq.to(U.dtype)).all().item():
                        raise FloatingPointError("Nonfinite/out-of-range lambda for score dtype; no coefficient clipping is applied.")

            #### PART 4: final plug-in PPAT estimate and its OWN variance estimate
            ms = torch.arange(1, n + 1, device=device, dtype=torch.float64).unsqueeze(0)
            y_history = torch.gather(ys, dim=1, index=I_history).to(torch.float64)
            c_history = cs[I_history]  # frozen full-pool-centered proxy at the queried indices

            # Eqs. (5)/(22), pp. 3/38: final-budget LURE weights, not equal weights or HT weights.
            Vs = 1.0 + ((N_Q - n) / (N_Q - ms)) * (
                1.0 / ((N_Q - ms + 1.0) * q_history) - 1.0)
            # Fit the endpoint coefficient on ALL n labels, including endpoints <100 or not %100==0.
            if proxy_constant:
                lambda_final = torch.zeros((N_NEW, 1), dtype=torch.float64, device=device)
            else:
                lambda_final = (Vs * y_history * c_history).mean(dim=1, keepdim=True) / proxy_second_moment

            # Eq. (8), p. 4, and Section E.5.4, p. 56: ONE final lambda for every residual.
            residuals = y_history - lambda_final * c_history
            thetahats = (Vs * residuals).mean(dim=1, keepdim=True)
            # Lemma E.4, p. 38: sum over s<m excludes the current residual.
            prefix_residuals = torch.cumsum(residuals, dim=1) - residuals
            A_ms = (residuals / q_history + prefix_residuals) / N_Q
            gammas = N_Q * (N_Q - n) / ((N_Q - ms) * (N_Q - ms + 1.0))
            # Corollary E.19, p. 59: sigmahat^2 estimates the CLT variance, not Var(thetahat).
            sigmahats_sq = ((gammas * (A_ms - thetahats)) ** 2).mean(dim=1, keepdim=True)

            # Endpoint checks before writing results (no change to the statistical estimator).
            torch.testing.assert_close(thetahats, (gammas * A_ms).mean(dim=1, keepdim=True),
                                       rtol=1e-9, atol=1e-11)
            if not (already_sampled.sum(dim=1) == n).all().item():
                raise RuntimeError("A target/seed did not acquire exactly n distinct labels.")
            if not (torch.isfinite(lambda_final).all() & torch.isfinite(thetahats).all()
                    & torch.isfinite(sigmahats_sq).all()).item():
                raise FloatingPointError("Nonfinite fitted PPAT output; nothing is silently clipped.")

            # Same confidence level, finite-pool target, and untruncated width/coverage definitions.
            z_score = norm.ppf(1 - (ALPHA / 2))
            ubs = thetahats + z_score * (torch.sqrt(sigmahats_sq) / (n ** 0.5))
            lbs = thetahats - z_score * (torch.sqrt(sigmahats_sq) / (n ** 0.5))
            widths = (ubs - lbs).cpu().numpy()
            coverages = ((lbs <= true_thetas) & (true_thetas <= ubs)).cpu().numpy()
            thetahats = thetahats.cpu().numpy()
            final_lambdas = lambda_final.cpu().numpy()

            # Same per-model/seed rows as PAI; no pooling across target means.
            results = [
                [dataset, budget, TAU, METHOD, model_idx, seed,
                 thetahats[model_idx][0], coverages[model_idx][0], widths[model_idx][0], n]
                for model_idx in range(N_NEW)]
            with open(log_path, "a", newline="") as file:
                csv.writer(file).writerows(results)
            with open(lambda_path, "a", newline="") as file:
                csv.writer(file).writerows([
                    [dataset, budget, TAU, METHOD, model_idx, seed, final_lambdas[model_idx][0], n]
                    for model_idx in range(N_NEW)])
            clear_output(wait=True)  # original notebook's output cleanup
